# Week 1 - Cleaning & Exploring the Applicant Dataset
**Intern:** Maryam Shaikh | **Program:** NextGen Learners - Data Science Internship

This notebook cleans a raw applicant dataset (`applicants_by_maryamshaikh.csv`) using only Pandas code, so anyone can re-run it from scratch. Every decision is explained in the markdown cell above the code.

## Step 1 - Setup
Import Pandas, the only library needed.

In [ ]:
import pandas as pd

## Step 2 - Load the raw data
Phone is read as text (`dtype=str`) so that leading zeros are never lost.

In [ ]:
source_file = 'applicants_by_maryamshaikh.csv'
df = pd.read_csv(source_file, dtype={'Phone': str})

## Step 3 - First look (the "before" snapshot)
I explore the data before touching it, so I know exactly what needs fixing.

In [ ]:
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
print(df.isnull().sum())
print("Total missing:", df.isnull().sum().sum())
print("Exact duplicate rows:", df.duplicated().sum())

**Observations:** The dataset has **88 rows and 7 columns**. Missing values: Email 9, University 11, Status 14 (**34 in total**). Applicant Name, Phone, Domain Applied and Application Date have none. There are **3 exact duplicate rows**, and Application Date is stored as text (`object`), not as a real date.

## Step 4 - Handle missing values (column by column)
- **Applicant Name:** critical identifier. A nameless application cannot be used for outreach, so such rows are dropped (none were missing here, but the rule stays in the pipeline).
- **University:** not essential to decide an application, so I fill with `'Not Specified'` instead of losing the whole row.
- **Status:** a blank status most likely means the application is not processed yet, so I fill with `'Under Review'`.
- **Email:** also important, but I keep the row and use a placeholder. **I fill it later, after removing duplicates** (see Step 5), otherwise all placeholder rows would look like duplicates of each other.
- **Domain Applied:** no missing values in this dataset. The pipeline still maps any blank to `'Not Specified'` as a safety net.

In [ ]:
df = df.dropna(subset=['Applicant Name'])
df['University'] = df['University'].fillna('Not Specified')
df['Status'] = df['Status'].fillna('Under Review')
print(df.isnull().sum())

## Step 5 - Remove duplicates
1. Remove **exact duplicate rows** first.
2. Then check **near-duplicates by Email**, because Email is a more reliable unique identifier than Name. I lowercase and strip the emails first so `Ali@x.com` and `ali@x.com` count as the same address.
3. Rows with a **missing** email are kept, since a missing email does not mean the person applied twice.
4. Only now do I fill the missing emails with the placeholder `not_provided@example.com`.

In [ ]:
before = len(df)
df = df.drop_duplicates()
print("Exact duplicates removed:", before - len(df))

df['Email'] = df['Email'].str.strip().str.lower()
before = len(df)
df = df[df['Email'].isna() | ~df.duplicated(subset='Email')]
print("Email duplicates removed:", before - len(df))

df['Email'] = df['Email'].fillna('not_provided@example.com')
print("Rows now:", len(df))

**Result:** 3 exact duplicates were removed. No real email duplicates were found, so no further rows were dropped. This keeps all applicants who simply left the email blank.

## Step 6 - Standardize text fields
- Strip extra spaces from names and use Title Case.
- Map every spelling of a domain (any casing) to one standard label with a dictionary.
- Status: strip spaces and use Title Case, so `selected`, `SELECTED` and `Selected` become one value.
- I print any value that the dictionary did not catch, instead of silently turning it into NaN.

In [ ]:
df['Applicant Name'] = df['Applicant Name'].str.strip().str.title()

domain_mapping = {
    'web development': 'Web Development',
    'web dev': 'Web Development',
    'webdev': 'Web Development',
    'data science': 'Data Science',
    'ai': 'Artificial Intelligence',
    'artificial intelligence': 'Artificial Intelligence',
    'graphic design': 'Graphic Design',
    'cyber security': 'Cyber Security',
    'cybersecurity': 'Cyber Security',
}
clean_domain = df['Domain Applied'].str.strip().str.lower()
mapped = clean_domain.map(domain_mapping)
print("Unmapped domain values:", clean_domain[mapped.isna() & clean_domain.notna()].unique())

df['Domain Applied'] = mapped.fillna('Not Specified')
df['Status'] = df['Status'].str.strip().str.title()

print(df['Domain Applied'].unique())
print(df['Status'].unique())

The list of unmapped values is empty, so every spelling was caught. Domain Applied now has exactly **5 clean tracks**: Web Development, Data Science, Artificial Intelligence, Graphic Design, Cyber Security.

## Step 7 - Fix data types
- **Application Date:** converted to a real datetime. `errors='coerce'` turns unreadable dates into `NaT` instead of crashing. The data has two formats: `2026-05-12` (year-month-day) and `12/05/2026` (day/month/year). I parse each format explicitly and combine them, because letting Pandas guess can read the same date two different ways (12 May vs 5 Dec).
- **Phone:** dashes and spaces removed. It stays as text so leading zeros are kept.

In [ ]:
iso_dates = pd.to_datetime(df['Application Date'], format='%Y-%m-%d', errors='coerce')
slash_dates = pd.to_datetime(df['Application Date'], format='%d/%m/%Y', errors='coerce')
df['Application Date'] = iso_dates.fillna(slash_dates)
print("Unreadable dates (NaT):", df['Application Date'].isnull().sum())

df['Phone'] = df['Phone'].str.replace('-', '', regex=False).str.replace(' ', '', regex=False)

df.info()

In [ ]:
df['Application Date'].value_counts()

## Step 8 - Data Quality Summary

**Starting dataset:** 88 rows, 7 columns

**Issues found:**
- 34 missing values: Email 9, University 11, Status 14
- 3 exact duplicate rows (no duplicate applications by Email)
- Domain Applied had many spelling/casing variations for only 5 real domains
- Application Date stored as text, not as a date type
- Phone numbers contained dash/space formatting; names had extra whitespace

**Actions taken:**
- Dropped rows with a missing Applicant Name (none found)
- Filled missing University with `Not Specified` and Status with `Under Review`
- Removed 3 exact duplicates; checked duplicates by Email (excluding missing emails) and found none
- Filled 9 missing emails with `not_provided@example.com` **after** deduplication, so those applicants are not lost
- Standardized Domain Applied with a mapping dictionary; names and Status to Title Case
- Converted Application Date to datetime (day-first) and cleaned Phone formatting

**Final dataset:** 85 rows, 7 columns, no missing values, no duplicate applicants, consistent formatting. (The placeholder email appears several times on purpose; it marks "email not given", not a duplicate.)

In [ ]:
print("Final shape:", df.shape)
print(df.isnull().sum())
real = df[df['Email'] != 'not_provided@example.com']
print("Duplicate real emails:", real['Email'].duplicated().sum())

## Step 9 - Export the cleaned dataset
`index=False` avoids adding an extra unnamed row-number column.

In [ ]:
output_file = 'applicants_cleaned_by_maryamshaikh.csv'
df.to_csv(output_file, index=False)
print("Saved:", output_file)